# 02. Dynamic Shapes, Mark Dynamic, and Recompilation Guards

- **Category:** 12_compilation_and_performance
- **Lineage & Citations:** Ansel et al. (2024 *PyTorch 2: Dynamic Python Bytecode Transformation and Graph Compilation*); JAX Dynamic Shapes & Shape Polymorphism (2024).
- **Purpose:** Formulate and evaluate symbolic dimension compilation: `SymInt` tracking, Dynamo shape guards, `torch._dynamo.mark_dynamic`, `dynamic=True`, dynamic masked reductions, power-of-2 shape bucketing, and dynamic RMSNorm without recompilation thrashing.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [01. Torch Compile Modes Backends and Graph Breaks](01_torch_compile_modes_backends_and_graph_breaks.ipynb) $\to$ `🔗 Downstream Syntheses:` [03. Torch FX Graph Capture and Custom Compiler Passes](03_torch_fx_graph_capture_and_custom_compiler_passes.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do not pass changing Python scalar ints (e.g. `epoch`, `batch_size`) directly into compiled functions without marking dynamic or wrapping in 0D tensors (forces recompilation on every step); avoid static maximum padding without bucketing.

## Mathematical Foundations & Governing Equations

1. **Symbolic Shape Tracking**: $\text{Shape}(X) = (s_0, s_1, D), \quad s_i \in \text{SymInt}$
2. **Dynamic Masked Mean**: $\bar{x}_b = \frac{\sum_{t=1}^L X_{b, t} \cdot M_{b, t}}{\max\left(1, \sum_{t=1}^L M_{b, t}\right)}$
3. **Power-of-2 Bucket Allocation**: $b^* = \arg\min \{b \in \mathcal{B} : b \ge L\}$
4. **Dynamic RMSNorm**: $y = \frac{x}{\sqrt{\frac{1}{D}\sum_{d=1}^D x_d^2 + \epsilon}} \odot W$

In [ ]:
import sys, time
from pathlib import Path
import torch
from safetensors.torch import load_file

r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
if str(r) not in sys.path: sys.path.insert(0, str(r))
w_path = r / "weights/12_compilation_and_performance/02_dynamic_shapes_mark_dynamic_and_recompilation_guards.safetensors"
weights = load_file(str(w_path))
print(f"Loaded {len(weights)} tensors from {w_path.name}")

In [ ]:
def dynamic_masked_pool(x, mask):
    mask_expanded = mask.unsqueeze(-1)
    sum_x = torch.sum(x * mask_expanded, dim=1)
    lengths = torch.sum(mask, dim=-1, keepdim=True)
    return sum_x / torch.clamp(lengths, min=1.0)

def bucket_allocation(seq_lens, bucket_boundaries):
    diffs = bucket_boundaries.unsqueeze(0) - seq_lens.unsqueeze(1)
    valid = diffs >= 0.0
    cost = torch.where(valid, diffs, torch.tensor(float("inf")))
    idx = torch.argmin(cost, dim=-1)
    return bucket_boundaries[idx]

def dynamic_rms_norm(x, weight, eps=1e-5):
    variance = torch.mean(x ** 2, dim=-1, keepdim=True)
    return (x / torch.sqrt(variance + float(eps))) * weight

In [ ]:
# Parity verification against golden reference tensors
mp = dynamic_masked_pool(weights["pool_x"], weights["pool_mask"])
assert torch.max(torch.abs(mp - weights["masked_pool_ref"])) < 1e-5

alloc = bucket_allocation(weights["seq_lens"], weights["buckets"])
assert torch.max(torch.abs(alloc - weights["allocated_buckets_ref"])) < 1e-5

rms = dynamic_rms_norm(weights["rms_x"], weights["rms_w"], eps=1e-5)
assert torch.max(torch.abs(rms - weights["dynamic_rms_ref"])) < 1e-5
print("All Dynamic Masked Pool, Bucket Allocation, and RMSNorm parity assertions passed!")

In [ ]:
# Benchmarking dynamic shape operators
t0 = time.perf_counter()
for _ in range(500):
    _ = dynamic_masked_pool(weights["pool_x"], weights["pool_mask"])
    _ = bucket_allocation(weights["seq_lens"], weights["buckets"])
    _ = dynamic_rms_norm(weights["rms_x"], weights["rms_w"], 1e-5)
lat_us = (time.perf_counter() - t0) / 500 * 1e6
print(f"Combined dynamic shape operators latency: {lat_us:.2f} us")

In [ ]:
# Diagnostic & Failure Mode: Recompilation Avoidance via Bucketing
# 1. Verify bucket assignments: sequence lengths are mapped to discrete enclosing boundaries
lens = weights["seq_lens"]
buckets = weights["buckets"]
for i, l in enumerate(lens):
    b = alloc[i].item()
    print(f"Sequence Length {l.item():.0f} -> Allocated Bucket {b:.0f} (Capacity utilization: {l.item()/b*100:.1f}%)")
    assert b >= l.item(), "Bucket must cover sequence length!"

# 2. Masked reduction: verify sample with length 1 only pools first element
first_elem = weights["pool_x"][2, 0]
pooled_elem = mp[2]
assert torch.max(torch.abs(first_elem - pooled_elem)) < 1e-5, "Single-element mask must match first token!"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Symbolic Dimension Compilation** (`SymInt`) compiles kernels with dynamic loop bounds, eliminating recompilation pauses.
2. **mark_dynamic** tags specific axes (batch, sequence) as dynamic while preserving static specialization for hidden feature widths.
3. **Power-of-2 Bucketing** limits the total space of compiled kernels to $\mathcal{O}(\log L_{\max})$ while bounding padding compute waste.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [02_dynamic_shapes_mark_dynamic_and_recompilation_guards.md](../../markdown/12_compilation_and_performance/02_dynamic_shapes_mark_dynamic_and_recompilation_guards.md)
- **PyTorch 2.x Module:** [02_dynamic_shapes_mark_dynamic_and_recompilation_guards.py](../../pytorch2/12_compilation_and_performance/02_dynamic_shapes_mark_dynamic_and_recompilation_guards.py)
- **Functional JAX Module:** [02_dynamic_shapes_mark_dynamic_and_recompilation_guards.py](../../jax/12_compilation_and_performance/02_dynamic_shapes_mark_dynamic_and_recompilation_guards.py)
- **Native Rust Candle Track:** [02_dynamic_shapes_mark_dynamic_and_recompilation_guards.rs](../../candle/12_compilation_and_performance/02_dynamic_shapes_mark_dynamic_and_recompilation_guards.rs)
- **Hardware Kernel (CPU / WGSL):** [02_dynamic_shapes_mark_dynamic_and_recompilation_guards.rs](../../kernels/12_compilation_and_performance/02_dynamic_shapes_mark_dynamic_and_recompilation_guards.rs)
- **Golden SafeTensors:** [02_dynamic_shapes_mark_dynamic_and_recompilation_guards.safetensors](../../weights/12_compilation_and_performance/02_dynamic_shapes_mark_dynamic_and_recompilation_guards.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 01. Torch Compile Modes Backends and Graph Breaks](01_torch_compile_modes_backends_and_graph_breaks.ipynb) | [Master Index](../../README.md#12_compilation_and_performance) | [Next: 03. Torch FX Graph Capture and Custom Compiler Passes](03_torch_fx_graph_capture_and_custom_compiler_passes.ipynb) &rarr;